In [ ]:
# pyright: reportUnusedImport=false, reportMissingImports=false, reportMissingModuleSource=false
import tensorflow as tf
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input, Lambda
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input # process images in exact fromat required for ResNet50

In [3]:

BASE_DIR = Path.cwd()
TRAIN_DIR = BASE_DIR / "dataset" / "train"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

In [ ]:

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    seed=100, # Ensures that the training and validation datasets are split in the same way each time you run the code
)

Found 270 files belonging to 2 classes.
Using 216 files for training.


In [5]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="validation",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    seed=100,
)

Found 270 files belonging to 2 classes.
Using 54 files for validation.


In [6]:
class_names = train_dataset.class_names
print("Classes:", class_names)

Classes: ['hotdog', 'not_hotdog']


In [7]:
for images, labels in train_dataset.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)


Image batch shape: (32, 224, 224, 3)
Label batch shape: (32, 1)


In [9]:

data_augmentation = Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
])


In [ ]:
train_dataset = (
    train_dataset
    .cache()
    .shuffle(100)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .cache()
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

In [ ]:

resnet_base = ResNet50(
    weights="imagenet",#Means ResNet50 already learned from the ImageNet dataset
    include_top=False, #Means: remove ResNet50’s original final classification layer.
    input_shape=(224, 224, 3),
)

In [12]:

resnet_base.trainable = False

In [13]:
model = Sequential([
    Input(shape=(224, 224, 3)),
    data_augmentation,
    Lambda(preprocess_input),
    resnet_base,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])


In [14]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_1 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         2,049 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,589,761 (89.99 MB)

 Trainable params: 2,049 (8.00 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [15]:

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=["accuracy"],
)


In [16]:
history = model.fit(
    train_dataset,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        EarlyStopping(
            monitor="val_accuracy",
            patience=5,
            restore_best_weights=True,
        )
    ],
)

Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.6435 - loss: 0.6773 - val_accuracy: 0.7037 - val_loss: 0.5015
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.7917 - loss: 0.3919 - val_accuracy: 0.7593 - val_loss: 0.4102
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.9028 - loss: 0.2980 - val_accuracy: 0.9259 - val_loss: 0.3449
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.9074 - loss: 0.2345 - val_accuracy: 0.8333 - val_loss: 0.3139
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.9213 - loss: 0.1960 - val_accuracy: 0.9074 - val_loss: 0.2752
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 988ms/step - accuracy: 0.9398 - loss: 0.1770 - val_accuracy: 0.9259 - val_loss: 0.2532
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.9676 - loss: 0.1287 - val_accuracy: 0.9259 - val_loss: 0.2298
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.9444 - loss: 0.1412 - val_accuracy: 0.9259 - val_loss: 0.2075


In [17]:
model.save(BASE_DIR / "hotdog_resnet50_classifier.keras")
print("Model saved as hotdog_resnet50_classifier.keras")

Model saved as hotdog_resnet50_classifier.keras
